# Lesson 2.4 — Built-in Objects & Rendering

**Phase 2: Chart Anatomy & Templates**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Access all built-in Helm objects in templates
- Use `.Release`, `.Chart`, `.Files`, and `.Capabilities` effectively
- Use `helm get` commands to inspect deployed resources
- Write a NOTES.txt that displays useful post-install guidance

---

## The Built-in Objects

> **Reference:** [Built-in Objects](https://helm.sh/docs/chart_template_guide/builtin_objects/) — https://helm.sh/docs/chart_template_guide/builtin_objects/

### `.Release` Object

| Field | Value Example | Description |
|---|---|---|
| `.Release.Name` | `my-app` | The release name given at `helm install` |
| `.Release.Namespace` | `production` | Kubernetes namespace |
| `.Release.IsInstall` | `true/false` | True on first install |
| `.Release.IsUpgrade` | `true/false` | True on upgrade or rollback |
| `.Release.Service` | `Helm` | Always "Helm" |
| `.Release.Revision` | `3` | Release revision number |

In [ ]:
# Create a chart that showcases all built-in objects
rm -rf /tmp/builtins && mkdir -p /tmp/builtins/templates

cat > /tmp/builtins/Chart.yaml << 'EOF'
apiVersion: v2
name: builtins
version: 0.1.0
appVersion: "2.5.1"
EOF

cat > /tmp/builtins/values.yaml << 'EOF'
message: "Hello from values"
EOF

# Template that demonstrates all built-in objects
cat > /tmp/builtins/templates/info.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: {{ .Release.Name }}-info
  namespace: {{ .Release.Namespace }}
data:
  # .Release built-ins
  releaseName: {{ .Release.Name }}
  releaseNamespace: {{ .Release.Namespace }}
  releaseService: {{ .Release.Service }}
  isInstall: {{ .Release.IsInstall | quote }}
  isUpgrade: {{ .Release.IsUpgrade | quote }}

  # .Chart built-ins (from Chart.yaml)
  chartName: {{ .Chart.Name }}
  chartVersion: {{ .Chart.Version }}
  chartAppVersion: {{ .Chart.AppVersion }}

  # .Values
  customMessage: {{ .Values.message | quote }}
EOF

helm template demo-release /tmp/builtins --namespace production

---

## `.Files` Object

Use `.Files` to embed non-template files (config files, scripts, certs) from the chart into templates:

In [ ]:
# Create a config file inside the chart (not in templates/)
mkdir -p /tmp/builtins/config
cat > /tmp/builtins/config/app.conf << 'EOF'
worker_processes auto;
error_log /var/log/nginx/error.log;
pid /run/nginx.pid;
EOF

# Template that reads the file using .Files.Get
cat > /tmp/builtins/templates/configfile.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: {{ .Release.Name }}-nginx-config
data:
  nginx.conf: |
    {{- .Files.Get "config/app.conf" | nindent 4 }}
EOF

helm template demo-release /tmp/builtins --show-only templates/configfile.yaml

---

## `.Capabilities` Object

Use `.Capabilities` to write templates that adapt to the Kubernetes version:

In [ ]:
cat > /tmp/builtins/templates/capabilities-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: {{ .Release.Name }}-caps
data:
  # .Capabilities.KubeVersion.Version returns the Kubernetes version string
  kubeVersion: {{ .Capabilities.KubeVersion.Version | default "unknown" | quote }}

  # Conditional based on K8s API availability
  # (In real templates: use this to choose between networking.k8s.io/v1 and extensions/v1beta1 for Ingress)
  # hasIngressV1: {{ .Capabilities.APIVersions.Has "networking.k8s.io/v1" | quote }}
EOF

helm template demo-release /tmp/builtins --show-only templates/capabilities-demo.yaml

---

## Writing NOTES.txt

`templates/NOTES.txt` is printed to the user after `helm install` or `helm status`. It should give actionable next steps:

In [ ]:
cat > /tmp/builtins/templates/NOTES.txt << 'EOF'
Thank you for installing {{ .Chart.Name }} version {{ .Chart.AppVersion }}!

Release name: {{ .Release.Name }}
Namespace:    {{ .Release.Namespace }}

To access the application:

  kubectl port-forward svc/{{ .Release.Name }} 8080:80 --namespace {{ .Release.Namespace }}
  Then open: http://localhost:8080

To check the release status:

  helm status {{ .Release.Name }} --namespace {{ .Release.Namespace }}

To upgrade the chart:

  helm upgrade {{ .Release.Name }} <chart-path> --namespace {{ .Release.Namespace }}
EOF

# Render NOTES.txt using helm template --show-only
helm template demo-release /tmp/builtins \
  --show-only templates/NOTES.txt \
  --namespace production

---

## Using `helm get` to Inspect Deployed Releases

After `helm install`, these commands let you inspect a live release:

In [ ]:
# These commands require a running release — shown here for reference
echo "== Commands for inspecting a deployed release =="
echo ""
echo "helm get manifest <release>   # Show deployed K8s YAML"
echo "helm get values <release>     # Show effective values (user-supplied)"
echo "helm get values <release> --all  # Show ALL values including defaults"
echo "helm get notes <release>      # Show the rendered NOTES.txt"
echo "helm get hooks <release>      # Show hook templates"
echo "helm get all <release>        # Everything above combined"

---

## Exercise

In [ ]:
# TASK 1: In /tmp/builtins/templates/info.yaml, add a line:
#         revision: (the .Release.Revision built-in)
#         Re-render and verify the value appears.
# TODO:


# TASK 2: Add a second config file to /tmp/builtins/config/settings.ini
#         with some key=value content, then modify configfile.yaml to
#         also include it as a second key in the ConfigMap.
# TODO:


# TASK 3: Write a NOTES.txt snippet that conditionally prints different
#         messages depending on whether this is an install vs upgrade.
#         (Hint: use .Release.IsInstall and .Release.IsUpgrade —
#          we cover if/else in Lesson 3.2, but try it here!)
# TODO:

echo "Exercises: implement the steps above"

---

## Summary

| Object | Key Fields |
|---|---|
| `.Release` | `.Name`, `.Namespace`, `.IsInstall`, `.IsUpgrade`, `.Revision` |
| `.Chart` | `.Name`, `.Version`, `.AppVersion` |
| `.Values` | All values from `values.yaml` and user overrides |
| `.Files` | `.Get`, `.GetBytes`, `glob` — access bundled non-template files |
| `.Capabilities` | `.KubeVersion`, `.APIVersions.Has` — adapt to cluster version |

- `templates/NOTES.txt` is rendered as a template and shown after install.
- Use `helm get` subcommands to inspect a deployed release in detail.

---

## Next Lesson

➡️ [Lesson 3.1 — Template Functions & Pipelines](../phase-3-advanced-templates/lesson-3.1-functions-pipelines.ipynb)